# Función de clasificación de tweets

Elabora una función en la que el usuario ingresa un tweet sin preprocesar y el sistema
lo clasifica en desastre real o no desastre.

**Recursos** El modelo y el vectorizador ya entrenados y guardados en la carpeta output.


## 1. Preprocesamiento

Estas funciones son idénticas a las usadas para entrenar el modelo. 

In [5]:
import re
import html
import string

import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

for pkg, path in [
    ("stopwords", "corpora/stopwords"),
    ("punkt", "tokenizers/punkt"),
    ("punkt_tab", "tokenizers/punkt_tab"),
    ("wordnet", "corpora/wordnet"),
    ("omw-1.4", "corpora/omw-1.4"),
]:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(pkg)

pd.set_option("display.max_colwidth", 200)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sofia\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\sofia\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [6]:
URL_RE      = re.compile(r"(https?://\S+|www\.\S+)")
MENTION_RE  = re.compile(r"@\w+")
HASHTAG_RE  = re.compile(r"#(\w+)")

EMOTICON_RE = re.compile(r"[:;=8][\-o\*']?[\)\]\(\[dDpP\\\|]+|<3|\^_\^|-_-|:'\(")

EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002190-\U000021FF"
    "\U00002B00-\U00002BFF"
    "\U0000FE0F"
    "]+",
    flags=re.UNICODE,
)
NUMBER_RE   = re.compile(r"\b\d+\b")
NON_ALNUM_RE = re.compile(r"[^a-z0-9\s]")
SPACES_RE   = re.compile(r"\s+")


CONTRACCIONES = [
    (re.compile(r"\bwon't\b"), "will not"),
    (re.compile(r"\bcan't\b"), "can not"),
    (re.compile(r"\bi'm\b"), "i am"),
    (re.compile(r"\blet's\b"), "let us"),
    (re.compile(r"n't\b"), " not"),
    (re.compile(r"'re\b"), " are"),
    (re.compile(r"'ll\b"), " will"),
    (re.compile(r"'ve\b"), " have"),
    (re.compile(r"'d\b"), " would"),
]

def limpiar_texto(texto: str) -> str:
    t = str(texto)
    t = html.unescape(t)
    t = t.lower()
    t = URL_RE.sub(" ", t)
    t = MENTION_RE.sub(" ", t)
    t = HASHTAG_RE.sub(r"\1", t)
    t = EMOJI_RE.sub(" ", t)
    for pat, repl in CONTRACCIONES:
        t = pat.sub(repl, t)
    t = t.replace("'", "")
    t = re.sub(r"\b(?!911\b)\d+\b", " ", t)
    t = NON_ALNUM_RE.sub(" ", t)
    t = SPACES_RE.sub(" ", t).strip()
    return t

In [7]:
STOP_EN = set(stopwords.words("english"))
STOP_EN -= {"no", "not", "nor"}

lem = WordNetLemmatizer()

def tokenizar(texto: str) -> list:
    tokens = word_tokenize(texto)
    tokens = [tok for tok in tokens if tok not in STOP_EN and len(tok) > 2 or tok == "911"]
    tokens = [lem.lemmatize(tok) for tok in tokens]
    return tokens

## 2. Cargar el modelo y el vectorizador entrenados

In [9]:
import joblib
from pathlib import Path

OUT = Path("../output")
modelo     = joblib.load(OUT / "modelo_final.pkl")
vectorizer = joblib.load(OUT / "vectorizer.pkl")

print("Modelo:     ", type(modelo).__name__)
print("Vectorizer: ", type(vectorizer).__name__,
      "| vocabulario:", len(vectorizer.vocabulary_), "n-gramas")

Modelo:      LogisticRegression
Vectorizer:  TfidfVectorizer | vocabulario: 8000 n-gramas


## 3. La función de clasificación

In [10]:
def preprocesar(texto: str) -> str:
    return " ".join(tokenizar(limpiar_texto(texto)))


def clasificar_tweet(texto: str) -> dict:
    texto_final = preprocesar(texto)

    if not texto_final:
        return {
            "tweet": texto,
            "texto_procesado": "",
            "clase": "No se puede clasificar (sin texto util tras la limpieza)",
            "target": None,
            "confianza": None,
        }

    X     = vectorizer.transform([texto_final])
    pred  = int(modelo.predict(X)[0])
    proba = modelo.predict_proba(X)[0]

    return {
        "tweet": texto,
        "texto_procesado": texto_final,
        "clase": "Desastre real" if pred == 1 else "No desastre",
        "target": pred,
        "confianza": round(float(proba[pred]), 3),
    }

## 4. Pruebas

In [11]:
ejemplos = [
    "Forest fire spreading fast near the town, everyone evacuate now!",
    "just had the best pizza of my life lol",
    "OMG the sky looks like it's on fire during this sunset",
    "BREAKING: 6.2 magnitude earthquake hits the coast, buildings collapsed",
    "I'm crying this movie destroyed me emotionally",
    "Call 911 there's smoke everywhere and people are trapped #fire",
]

pd.DataFrame([clasificar_tweet(t) for t in ejemplos])

,tweet,texto_procesado,clase,target,confianza
0,"Forest fire spreading fast near the town, everyone evacuate now!",forest fire spreading fast near town everyone evacuate,Desastre real,1,0.839
1,just had the best pizza of my life lol,best pizza life lol,No desastre,0,0.821
2,OMG the sky looks like it's on fire during this sunset,omg sky look like fire sunset,Desastre real,1,0.526
3,"BREAKING: 6.2 magnitude earthquake hits the coast, buildings collapsed",breaking magnitude earthquake hit coast building collapsed,Desastre real,1,0.893
4,I'm crying this movie destroyed me emotionally,cry movie destroyed emotionally,No desastre,0,0.822
5,Call 911 there's smoke everywhere and people are trapped #fire,call 911 there smoke everywhere people trapped fire,Desastre real,1,0.626


In [ ]:
#se responde en el buscador de arriba

tweet = input("Escribe un tweet: ")
resultado = clasificar_tweet(tweet)
(f"{resultado['clase']}  (confianza: {resultado['confianza']})")


'\nDesastre real  (confianza: 0.765)'